# Borrowed AWS session and optional ADA · 2.1.9

Input → prepared session, region and model. Process → runtime validates, Agent compiles and ADA inspects without resolving credentials. Output → safe configuration. This is an offline configuration check, not IAM validation. No installations or service calls occur. The local SDK and ADA add-on are required. All declarations are visible.

In [ ]:
"""Borrow an AWS session: configuration only, not IAM or ADA certification."""

import json

import boto3
import agentic_systems as toolkit
import agentic_systems_ada as ada


# This explicit offline fixture avoids looking up credentials or metadata.
# In ADA, use the session supplied by your platform instead of fixture keys.
session = boto3.Session(
    region_name="us-east-2",
    aws_access_key_id="offline-fixture",
    aws_secret_access_key="offline-fixture",
)
connection = toolkit.runtime(
    provider="bedrock-runtime", model="qwen-model-enabled-in-your-account",
    session=session,
)
agent = toolkit.Agent("aws_worker", runtime=connection)
program = agent.compile()


def run():
    # compile prepares; it does not create clients or invoke a model.
    assert connection.region == "us-east-2"
    assert "session" not in connection.model_dump()
    report = ada.diagnose(session=session)
    assert not report["services_checked"] and not report["credentials_resolved"]
    return {
        "input": "Borrowed boto3 Session, model and region",
        "process": "Validate connection -> compile Agent -> inspect configuration without services",
        "output": {"runtime": connection.inspect(), "diagnostic": report},
        "validation_scope": "configuration-offline-passed",
        "pending": ["real AWS IAM", "real ADA IAM", "model access and business acceptance"],
        "inference_requests": 0,
    }

In [ ]:
report = run()
print(json.dumps(report, ensure_ascii=False, indent=2))
assert report["validation_scope"] == "configuration-offline-passed"

In your environment, replace only the fixture session with the platform-prepared session. Never paste credentials into the notebook. API keys and IAM sessions cannot be combined; conflicts are rejected. The resource stays out of state and results. Its owner is responsible for external concurrent use. Real IAM/ADA access remains unverified. [Limits and execution](README.md).